# C2.2 · Model-layer research

**Function C — Red Teaming and Security Research with AI → Security Research with AI**  ·  *Security of AI*

Builds on **[C2.1 · What research means in a CISO org](https://spbreed.github.io/cyber-commons/lessons/C2.1.html)**.

| | |
|---|---|
| Tools used | garak, Llama 3.3, GLM-4.6, Kimi K2, Claude Opus 5 |

## What this lesson is

**What it covers.** Run a jailbreak taxonomy across Llama, GLM and Kimi and chart where they differ.

**Why a security engineer needs it.** Model cards read credulously. The control it builds is: adversarial robustness, jailbreak taxonomy, refusal analysis, capability elicitation.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

The model layer is the one place where the same input legitimately produces different output, which makes every naive experiment on it unrepeatable. Method is not a formality here; it is the only thing separating a finding from a coincidence.

> **At CyberTravels.** “CyberTravels refunded a booking when I asked” is one attempt. The rate, with an interval, is what changes when the provenance control ships and what tells you the change was real.

## 2 · The framework

```
   same prompt, same model, three runs, three outputs
        |
        v
   +--------------------------------------------+
   | n trials . fixed seeds . a control arm     |
   | report a RATE with an interval, not a case |
   +--------------------------------------------+

   without method, a finding and a coincidence look identical
```

Model-layer research means treating the model as an object of study rather than
a demo subject. The discipline is one rule: **report rates, not anecdotes.**

"I got it to do X" is not a result. Language models are stochastic; with enough
attempts you can get almost anything once. The result is the *rate*, with an
interval, because the rate is what changes when a mitigation lands and the
interval is what tells you whether the change was real.

This matters practically. A mitigation that moves a technique from 62% to 48%
sounds like progress. With n=20 the confidence intervals overlap so heavily that
you have demonstrated nothing, and you are about to tell a board you reduced
risk by 23%.

## 3 · The control — compute the sample size before you run

The question is not "how many attempts should I do?" It is: **how small an effect do I need to be able to detect?**

## 4 · The procedure, as a skill

One success is an anecdote. The skill runs each technique enough times to classify it — not reproduced, flaky, reproducible — and then computes the sample size the before-and-after comparison actually needed, before running it rather than after.

In [ ]:
# skills/research/technique-reproducibility-test/SKILL.md — embedded verbatim from the repository.
# This is the file itself, not a paraphrase of it.
SKILL_MD = r"""---
name: technique-reproducibility-test
description: >-
  Run a technique enough times to say whether it reproduces, and compute the
  sample size a before-and-after comparison actually needed before claiming a
  control worked. Use when a jailbreak "works", or when a fix is declared
  effective from a handful of trials.
allowed-tools: Read, Grep, Glob
---

# One success is an anecdote; the interval is the result

Model-layer research is statistical whether or not anybody does the statistics.
A technique that succeeds once may not reproduce; a control that appears to
help at n=20 has an interval overlapping the baseline. Both errors are avoided
by the same discipline — report a rate with an interval, and compute the sample
size before running the comparison.

## When to use this

Any claim about model behaviour: a technique that works, a control that helps, a
model that is safer than another.

## Procedure

**1 — Define success mechanically.** A string, a state, a check — something a
script decides. "The model complied" judged by reading is not reproducible
between two people.

**2 — Run enough trials to produce a rate, and hold the conditions fixed.**
Same model version, same temperature, same prompt. Record the version: a rate
without one is unrepeatable by construction.

**3 — Classify the technique honestly.** Not reproduced, flaky, or reproducible.
Flaky is a real and common answer and it deserves the word rather than a
rounded-up rate.

**4 — Compute the required sample size before comparing.** From the baseline
rate, the effect you would care about, and the power you want. Then run that
many. Doing this afterwards produces the number that makes the result you got
look significant.

**5 — Report intervals, and say when they overlap.** Show the same true effect
at n=20, n=100 and n=1000 if you need to make the point: the effect did not
change, the ability to see it did.

## Output contract

```json
{
  "success_criterion": "str",
  "conditions": {"model": "str", "version": "str", "temperature": 0.0},
  "techniques": [{"name": "str", "trials": 0, "rate": 0.0, "interval": [0.0, 0.0],
                  "verdict": "not reproduced|flaky|reproducible"}],
  "comparison": {"before": 0.0, "after": 0.0, "n": 0, "required_n": 0, "separated": false}
}
```

## Failure modes

- **Reporting a rate with no model version.** Nobody can repeat it.
- **Computing the sample size afterwards.** That is choosing the number that
  fits.
- **Rounding flaky up to works.** It is the finding, not a rough edge.
"""

In [ ]:
# Execute the skill above, using the shared runtime rather than a copy.
import glob, os, shutil, sys

# Make the shared runtime importable, then import it. On Kaggle an attached
# kernel is mounted as __script__.py — not on sys.path and not named after the
# kernel — so copy it to the name it is imported by. Locally it is already a
# file of that name in the repository.
_k = glob.glob("/kaggle/input/**/cyber-commons-skill-runtime/__script__.py", recursive=True)
if _k:
    shutil.copy(_k[0], "cyber_commons_skill_runtime.py")
sys.path[:0] = [".", "skills/_runtime", "../skills/_runtime", "../../skills/_runtime"]

from cyber_commons_skill_runtime import run_skill

# Split skills/research/technique-reproducibility-test/SKILL.md into the two halves an agent uses —
# the frontmatter it routes on, and the body it follows.
meta, body = run_skill(SKILL_MD)

In [ ]:
# skills/research/technique-reproducibility-test/scripts/technique_reproducibility_test.py — embedded verbatim from the repository.
# This is the skill's own script, not a paraphrase of it.
#!/usr/bin/env python3
"""Run a technique enough times to know whether it reproduces, and compute the sample size the comparison actually needed.

This is the executable half of the `technique-reproducibility-test` skill: the check the
SKILL.md next to it describes, run against a synthetic CyberTravels
estate so two runs can be diffed and the result argued with.

Standard library only, and deterministic, so it runs on a Kaggle
kernel with the internet switched off.
"""

import random

def trial(effect, n=200, seed=7):
    """Run a stochastic effect n times; report the rate with a 95% interval."""
    rng = random.Random(seed)
    hits = sum(effect(rng) for _ in range(n))
    rate = hits / n
    half = 1.96 * ((rate * (1 - rate) / n) ** 0.5) if n else 0.0
    lo, hi = round(max(rate - half, 0), 3), round(min(rate + half, 1), 3)
    verdict = ("reproducible" if lo > 0.5 else
               "flaky" if hi > 0.05 else "not reproduced")
    return {"n": n, "hits": hits, "rate": round(rate, 3), "ci95": (lo, hi),
            "verdict": verdict}

# ground-truth landing probabilities for three injection techniques
TECHNIQUES = {"direct override": 0.05, "context reframe": 0.35, "task nesting": 0.62}

print(f"{'technique':20s}{'rate':>7}{'ci95':>18}  verdict")
print("-" * 60)
for name, p in TECHNIQUES.items():
    r = trial(lambda rng, p=p: rng.random() < p, n=200)
    print(f"{name:20s}{r['rate']:>7.3f}{str(r['ci95']):>18}  {r['verdict']}")
print("\n'It worked' is true for all three. Only one is reproducible.")

def compare(before_p, after_p, n, seed=11):
    b = trial(lambda rng: rng.random() < before_p, n=n, seed=seed)
    a = trial(lambda rng: rng.random() < after_p,  n=n, seed=seed + 1)
    overlap = a["ci95"][1] >= b["ci95"][0]
    return b, a, overlap

print(f"{'n':>6}{'before':>18}{'after':>18}  conclusion")
print("-" * 68)
for n in (20, 100, 1000):
    b, a, overlap = compare(0.62, 0.48, n)
    concl = "NOT demonstrated" if overlap else "improvement holds"
    print(f"{n:>6}{str(b['ci95']):>18}{str(a['ci95']):>18}  {concl}")
print("\nThe true effect is identical in all three rows. Only sample size changed.")
print("At n=20 you would report a 23% reduction you cannot support.")

def required_n(p_before, p_after, power_z=1.96):
    """Rough two-proportion sample size for a 95% interval that separates."""
    p = (p_before + p_after) / 2
    diff = abs(p_before - p_after)
    if diff == 0: return float("inf")
    return int((2 * power_z ** 2 * p * (1 - p)) / (diff ** 2)) + 1

print(f"{'effect you want to detect':34s}{'n required':>11}")
print("-" * 47)
for before, after in ((0.62, 0.10), (0.62, 0.31), (0.62, 0.48), (0.62, 0.58)):
    print(f"{f'{before:.0%} → {after:.0%}':34s}{required_n(before, after):>11}")
print("\nDetecting a halving is cheap. Detecting a 14-point move is not, and")
print("detecting a 4-point move is a research project in itself.")

n_needed = required_n(0.62, 0.48)
b, a, overlap = compare(0.62, 0.48, n_needed)
print(f"\nre-run at the computed n={n_needed}: "
      f"before {b['ci95']}, after {a['ci95']}, overlap={overlap}")

# Verify: the honest reporting template.
def report(technique, before, after, n):
    b, a, overlap = compare(before, after, n)
    return (f"{technique}\n"
            f"   before  {b['rate']:.2f} (95% CI {b['ci95']}, n={n})\n"
            f"   after   {a['rate']:.2f} (95% CI {a['ci95']}, n={n})\n"
            f"   verdict {'no demonstrated change — intervals overlap' if overlap else 'reduction demonstrated'}")

print(report("task nesting, after provenance mitigation", 0.62, 0.48, 20))
print()
print(report("task nesting, after provenance mitigation", 0.62, 0.48, 1000))

## What you just proved

Direct override is not reproduced, context reframe is flaky, task nesting is reproducible. The before/after comparison shows overlapping intervals at n=20 and n=100 and separation at n=1000, for an identical true effect. Sample-size calculation shows detecting 62%→48% needs roughly 200 trials while 62%→58% needs thousands.

## Your turn

Take the last jailbreak or injection result your team reported. Ask for n. If the answer is a single-digit number or 'we tried it a few times', the finding is real but the number attached to it is not.

---

**Next → [C2.3 · Weight-level techniques](https://spbreed.github.io/cyber-commons/lessons/C2.3.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/C2.2.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/C2.2.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*